# 02 — Pipeline Walkthrough
Run each stage manually on a small synthetic dataset and inspect the intermediate artefacts.

In [ ]:
import sys, os, numpy as np
sys.path.insert(0, os.path.abspath('..'))
from src.data_simulation import SimulationConfig, simulate_dataset
from src import (stage1_ld_preprocessing as s1, stage2_weak_effect_screening as s2,
                 stage3_interaction_aware_selection as s3, stage4_representation_learning as s4,
                 stage5_biological_validation as s5)
X, y, truth = simulate_dataset(SimulationConfig(n_individuals=800, n_snps=2500, seed=1))
print('input:', X.shape)

In [ ]:
k1, info1 = s1.run_stage1(X, y, s1.Stage1Config())
print('after Stage1:', k1.sum())
k2, info2 = s2.run_stage2(X[:, k1], y, truth['block_assignment'][k1], s2.Stage2Config())
print('after Stage2:', k2.sum())

In [ ]:
Xs2 = X[:, k1][:, k2]
k3, info3 = s3.run_stage3(Xs2, y, s3.Stage3Config(reliefF_topk=80, xgb_topk=80, seed=1))
print('after Stage3:', k3.sum())

In [ ]:
Xs3 = Xs2[:, k3]
contrib, Z, info4 = s4.run_stage4(Xs3, s4.Stage4Config(epochs=20, hidden_dim=64, bottleneck_dim=8, seed=1), y=y)
print('AE embedding shape:', Z.shape, ' top-5 contributing SNPs:', np.argsort(contrib)[-5:])